# 🇧🇩 BUET BanglaBERT Full End-to-End Fine-Tuning Suite
### Fine-Tuning `csebuetnlp/banglabert` (110M Params) for Multilingual Telecom VoC
**Grameenphone Ltd. (MyGP) 🔵 | Robi Axiata Limited (MyRobi) 🔴 | Banglalink (MyBL) 🟠**

---

### 🎯 Purpose & Scientific Context
In earlier baseline experiments, BUET BanglaBERT was tested as a **Frozen Feature Extractor** (where its 12 transformer layers were frozen with `torch.no_grad()`, and only a shallow 2-layer MLP was trained on static 768-dim embeddings). That setup scored **74.88% Accuracy / 0.6253 Macro-F1**, trailing our Classical Soft-Voting Ensemble (87.5%).

This notebook performs **Full End-to-End Fine-Tuning**:
- Gradients backpropagate through all 12 transformer layers (110 million parameters).
- The self-attention mechanisms adapt to Romanized Banglish (*"mb kete nilo"*, *"otp ashe na"*), mixed Bengali/English syntax, and telecom complaint intent.
- Covers both **Operational Categories (5 Classes)** and **Sentiment Analysis (3 Classes)**.

> ⚡ **Colab GPU Setup**: Go to **Runtime** ➔ **Change runtime type** ➔ **T4 GPU** ➔ **Save** before running.


In [ ]:
# 1. Verify GPU Setup
import torch

print("=" * 60)
print(f"PyTorch Version : {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device   : {device.type.upper()}")

if torch.cuda.is_available():
    print(f"GPU Hardware    : {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print("✓ T4 GPU acceleration is active and ready for fine-tuning!")
else:
    print("⚠️ Warning: Running on CPU. Please switch to GPU (Runtime > Change runtime type > T4 GPU).")
print("=" * 60)


In [ ]:
# 2. Clone Repository & Install Required Packages
import os
from pathlib import Path

if not Path("data/processed").exists():
    print("Cloning telecom VoC repository...")
    !git clone https://github.com/rh61632/telecom-Voice-of-Customer_intelligence.git
    %cd telecom-Voice-of-Customer_intelligence
else:
    print("Pulling latest updates...")
    !git pull

print("
Installing Hugging Face & training dependencies...")
!pip install -q transformers accelerate datasets scikit-learn pandas numpy
print("✓ Setup complete!")


---
## 📦 Part 1: Dataset Inspection ($N = 4,500$ Ground Truth Reviews)


In [ ]:
# 3. Load & Inspect Multi-Operator Training Dataset
import pandas as pd
import numpy as np

files = [
    "data/processed/mygp_classified_reviews.csv",
    "data/processed/myrobi_classified_reviews.csv",
    "data/processed/mybl_classified_reviews.csv",
]

df = pd.concat([pd.read_csv(f) for f in files if Path(f).exists()], ignore_index=True)
df = df.dropna(subset=["review_text", "sentiment", "category"]).copy()
df["review_text"] = df["review_text"].astype(str).str.strip()
df = df[df["review_text"].str.len() > 0]
df = df[df["category"] != "Unclassified"].reset_index(drop=True)

print("=" * 65)
print(f"Total Usable Ground-Truth Reviews : {len(df):,d}")
print(f"Operators Covered                 : {df["operator"].unique().tolist()}")
print("=" * 65)

print("
📊 Sentiment Distribution:")
print(df["sentiment"].value_counts(normalize=True).mul(100).round(2).astype(str) + "%")

print("
🏷️ Operational Category Distribution:")
print(df["category"].value_counts(normalize=True).mul(100).round(2).astype(str) + "%")


---
## 🧠 Part 2: Full Fine-Tuning Execution
We run the production fine-tuning script `dl/finetune_banglabert.py` which:
- Tokenizes via `csebuetnlp/banglabert` tokenizer (max length 96).
- Applies mixed-precision (`torch.amp.autocast`) for fast GPU execution.
- Computes inverse-frequency class weights for balanced Cross-Entropy loss.
- Uses AdamW with linear warmup learning rate schedule (`lr = 2e-5`).
- Evaluates 5-Fold Stratified Cross-Validation + trains a final production model.


In [ ]:
# 4. Launch Full End-to-End Fine-Tuning (Both Sentiment & Category)
# Runs 5-Fold Stratified CV and exports trained models to dl/models/
!python dl/finetune_banglabert.py --task both --epochs 4 --batch_size 32 --lr 2e-5


---
## 📊 Part 3: View Benchmark Results & Multi-Paradigm Comparison


In [ ]:
# 5. Display Fine-Tuned Benchmark Report
report_path = Path("dl/reports/banglabert_finetuning_report.txt")
if report_path.exists():
    print(report_path.read_text())
else:
    print("Report file not generated yet. Please run the fine-tuning cell above.")


---
## 🔮 Part 4: Live Interactive Inference Tester
Test any Bengali, Banglish, or English telecom review with your newly fine-tuned models!


In [ ]:
# 6. Live Interactive Predictor using Fine-Tuned Weights
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

SENT_MAP = {0: "Negative", 1: "Neutral", 2: "Positive"}
CAT_MAP = {
    0: "General Appreciation / Other",
    1: "Offers & Data Packs",
    2: "App Login & Technical Bugs",
    3: "Network Speed & 4G Latency",
    4: "Billing & Airtime Deductions",
}

sent_model_dir = "dl/models/banglabert_finetuned_sentiment"
cat_model_dir = "dl/models/banglabert_finetuned_category"

if Path(sent_model_dir).exists() and Path(cat_model_dir).exists():
    tokenizer = AutoTokenizer.from_pretrained(sent_model_dir)
    sent_model = AutoModelForSequenceClassification.from_pretrained(sent_model_dir).to(device)
    cat_model = AutoModelForSequenceClassification.from_pretrained(cat_model_dir).to(device)
    sent_model.eval()
    cat_model.eval()

    def predict_review(text):
        enc = tokenizer(text, truncation=True, max_length=96, return_tensors="pt").to(device)
        with torch.no_grad():
            s_out = sent_model(**enc)
            s_prob = F.softmax(s_out.logits, dim=-1).squeeze().cpu().numpy()
            s_pred = SENT_MAP[int(s_prob.argmax())]
            s_conf = s_prob.max()

            c_out = cat_model(**enc)
            c_prob = F.softmax(c_out.logits, dim=-1).squeeze().cpu().numpy()
            c_pred = CAT_MAP[int(c_prob.argmax())]
            c_conf = c_prob.max()

        print("=" * 65)
        print(f"INPUT REVIEW: \"{text}\"")
        print(f"PREDICTED SENTIMENT : {s_pred} ({s_conf*100:.1f}% confidence)")
        print(f"PREDICTED CATEGORY  : {c_pred} ({c_conf*100:.1f}% confidence)")
        print("=" * 65)

    # Test sample real-world reviews
    predict_review("MB kete nilo taka o nai net o slow")
    predict_review("Update er por theke OTP ashe na login hocche na")
    predict_review("Onk shundor app shob offer e discount pai")
    predict_review("4G connection thakleo kono kaje ashe na")
else:
    print("Fine-tuned models not found. Please run cell 4 first.")


---
## 💾 Part 5: Package & Download Trained Checkpoints


In [ ]:
# 7. Zip Fine-Tuned Checkpoints and Download to Browser
import shutil
import zipfile
from google.colab import files

zip_name = "banglabert_finetuned_models.zip"

print("Packaging fine-tuned BanglaBERT checkpoints...")
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for base_dir in ["dl/models/banglabert_finetuned_sentiment", "dl/models/banglabert_finetuned_category"]:
        p = Path(base_dir)
        if p.exists():
            for f in p.rglob("*"):
                if f.is_file():
                    z.write(f, arcname=str(f.relative_to("dl/models")))
                    print(f"  + Added: {f.name}")

if Path(zip_name).exists():
    print(f"\n✓ Created archive: {zip_name} ({Path(zip_name).stat().st_size / 1e6:.1f} MB)")
    print("Triggering browser download...")
    files.download(zip_name)
else:
    print("No model directories found to zip.")
